In [1]:
!pip install -q -U accelerate peft trl bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 17.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import gc
import json
import time
from copy import deepcopy

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

In [5]:
import os
from dotenv import load_dotenv

# 1. Load các biến từ tệp .env (nếu có)
load_dotenv()

# 2. Khai báo token trực tiếp (chỉ dùng nếu .env không hoạt động hoặc chưa cấu hình)
HF_TOKEN_DIRECT = "hf_" 

# 3. Logic kiểm tra: Ưu tiên lấy từ môi trường trước
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    print("✅ HF_TOKEN loaded successfully from environment (.env).")
else:
    # Nếu không tìm thấy trong environment, sử dụng giá trị khai báo trực tiếp
    HF_TOKEN = HF_TOKEN_DIRECT
    if HF_TOKEN and HF_TOKEN != "hf_":
        print("ℹ️ Using HF_TOKEN from direct variable.")
    else:
        print("❌ Error: No HF_TOKEN found in .env or direct variable!")
        HF_TOKEN = None

ℹ️ Using HF_TOKEN from direct variable.


In [6]:
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
CACHE_DIR = "./cache"
OUTPUT_ROOT = "./out_full_trl"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [7]:
# --- Hyperparameters ---
MAX_SEQ_LENGTH = 128
SEED = 42

# --- LoRA Configuration ---
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

In [8]:
# --- Device & Dtype Setup ---
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

In [9]:
# --- Training Arguments ---
COMMON_TRAIN_KWARGS = dict(
    num_train_epochs=1,
    max_steps=200,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    optim="adamw_torch",
    report_to="none",
    disable_tqdm=False,
    load_best_model_at_end=False,
    seed=SEED,
)

# --- Results Placeholders ---
histories = {}
run_stats = {}
infer_results = {}

In [10]:
from datasets import load_dataset

# --- 1. Load Dataset ---
raw_datasets = load_dataset("thainq107/Vi-Alpaca-Preference")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/506 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/61.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.95M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/65017 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [11]:
# --- 2. Define Formatting Function ---
# Chuyển đổi định dạng Alpaca sang ChatML (OpenAI format) 
# Định dạng này giúp Llama-3-Instruct hiểu rõ vai trò System/User/Assistant
def row_to_messages(example):
    return {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": example["chosen"]},
        ]
    }


In [12]:
# --- 3. Map & Clean Columns ---
# Chúng ta giữ lại cột 'messages' và loại bỏ các cột cũ để tránh xung đột
train_ds = raw_datasets["train"].map(
    row_to_messages,
    remove_columns=raw_datasets["train"].column_names,
    desc="Formatting train set"
)

eval_ds = raw_datasets["test"].map(
    row_to_messages,
    remove_columns=raw_datasets["test"].column_names,
    desc="Formatting eval set"
)

# --- 4. Subsampling (Tối ưu hóa thời gian huấn luyện) ---
TRAIN_SUBSAMPLE = 4000
EVAL_SUBSAMPLE = 800

if TRAIN_SUBSAMPLE:
    train_ds = train_ds.shuffle(seed=SEED).select(
        range(min(TRAIN_SUBSAMPLE, len(train_ds)))
    )

if EVAL_SUBSAMPLE:
    eval_ds = eval_ds.shuffle(seed=SEED).select(
        range(min(EVAL_SUBSAMPLE, len(eval_ds)))
    )

# --- 5. Prepare Inference Samples ---
# Trích xuất một vài mẫu để kiểm tra mô hình trước/sau khi train
INFER_SAMPLES = [
    {"question": eval_ds[i]["messages"][1]["content"]}
    for i in range(3)
]

print(f"✅ Setup complete! Train size: {len(train_ds)} | Eval size: {len(eval_ds)}")

Formatting train set:   0%|          | 0/65017 [00:00<?, ? examples/s]

Formatting eval set:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Setup complete! Train size: 4000 | Eval size: 800


In [13]:
print(INFER_SAMPLES)

[{'question': 'Xu hướng gần đây là gì?'}, {'question': 'Đưa ra một số dữ liệu ví dụ, phân loại dữ liệu thành các cụm.\n\n[{name: "John"}, {name: "Sara"}, {location: "New York"}, {location: "Washington DC"}]'}, {'question': 'Đưa ra một vật dụng hàng ngày, xây dựng một phép ẩn dụ về nó.\n\ncửa'}]


In [15]:
def clear_gpu():
    """Giải phóng bộ nhớ GPU và dọn rác hệ thống."""
    gc.collect()
    torch.cuda.empty_cache()

def apply_chat(example, tok):
    """
    Áp dụng Chat Template của mô hình vào dữ liệu thô.
    Biến đổi danh sách tin nhắn thành một chuỗi văn bản duy nhất 
    mà mô hình có thể hiểu được.
    """
    msgs = example["messages"]
    # Kiểm tra nếu msgs là một list các dict (một hội thoại đơn)
    if isinstance(msgs[0], dict):
        return [tok.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )]
    # Nếu là list của list (nhiều hội thoại)
    return [tok.apply_chat_template(
        m, tokenize=False, add_generation_prompt=False
    ) for m in msgs]

def param_stats(model):
    """Tính toán số lượng tham số tổng và tham số có thể huấn luyện."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total": total,
        "trainable": trainable,
        "ratio_pct": 100 * trainable / total
    }

def save_run(output_dir, label, framework, mode, seconds, stats):
    """Lưu lại thông tin chi tiết của lần chạy vào file JSON."""
    os.makedirs(output_dir, exist_ok=True)
    stat = {
        "run": label,
        "framework": framework,
        "mode": mode,
        "train_seconds": seconds,
        **stats,
    }
    # Cập nhật vào dictionary toàn cục (nếu có)
    run_stats[label] = stat
    with open(os.path.join(output_dir, "run_stats.json"), "w") as f:
        json.dump(stat, f, indent=2)

def extract_answer(full_text):
    """Trích xuất câu trả lời của Assistant từ chuỗi văn bản sinh ra."""
    lower = full_text.lower()
    # Tìm vị trí xuất hiện cuối cùng của từ khóa "assistant"
    idx = lower.rfind("assistant")
    if idx != -1:
        # Cắt lấy phần văn bản sau chữ "assistant"
        return full_text[idx + len("assistant"):].strip().lstrip(":\n ")
    return full_text.strip()

In [16]:
def train_trl(qlora: bool, output_dir: str, label: str):
    clear_gpu()
    print(f"\n=== {label} ===")

    # 1. Khởi tạo Tokenizer
    tok = AutoTokenizer.from_pretrained(
        MODEL_ID,
        cache_dir=CACHE_DIR,
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    # 2. Khởi tạo Model (Phân nhánh QLoRA và LoRA thường)
    if qlora:
        # Cấu hình nén mô hình 4-bit
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=DTYPE,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_cfg,
            device_map="auto",
            cache_dir=CACHE_DIR,
            trust_remote_code=True,
            token=HF_TOKEN,
        )
        # Chuẩn bị model để huấn luyện ở chế độ k-bit (tiết kiệm VRAM)
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )
    else:
        # Load model ở độ chính xác bình thường (DTYPE)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=DTYPE,
            device_map="auto",
            cache_dir=CACHE_DIR,
            trust_remote_code=True,
            token=HF_TOKEN,
        )

    # 3. Cấu hình LoRA
    model.config.use_cache = False  # Tắt cache khi training để tiết kiệm memory
    model = get_peft_model(
        model,
        LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=TARGET_MODULES,
        )
    )

    # In thống kê tham số
    ps = param_stats(model)
    print(f"trainable={ps['trainable']:,} ({ps['ratio_pct']:.3f}%)")

    # 4. Thiết lập tham số huấn luyện
    kw = dict(COMMON_TRAIN_KWARGS)
    # Nếu dùng QLoRA thì nên dùng optimizer paged để tối ưu VRAM tốt hơn
    kw["optim"] = "paged_adamw_8bit" if qlora else "adamw_torch"

    args = SFTConfig(
        output_dir=output_dir,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field=None, # Chúng ta dùng formatting_func nên để None
        **kw
    )

    # 5. Khởi tạo Trainer
    trainer = SFTTrainer(
        model=model,
        processing_class=tok,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        formatting_func=lambda ex: apply_chat(ex, tok),
    )

    # 6. Bắt đầu huấn luyện và lưu log
    t0 = time.perf_counter()
    trainer.train()
    elapsed = time.perf_counter() - t0
    print(f"[Done] {elapsed:.1f}s")

    # Lưu lại Adapter (không lưu toàn bộ model để tiết kiệm dung lượng)
    adapter_dir = os.path.join(output_dir, "adapter")
    model.save_pretrained(adapter_dir)
    tok.save_pretrained(adapter_dir)

    # Lưu lịch sử log
    log = deepcopy(trainer.state.log_history)
    with open(os.path.join(output_dir, "log_history.json"), "w") as f:
        json.dump(log, f, indent=2)

    histories[label] = log
    save_run(
        output_dir, label, "trl",
        "qlora" if qlora else "lora",
        elapsed, ps
    )

    # Dọn dẹp để giải phóng VRAM cho lần chạy sau
    del trainer, model
    clear_gpu()

In [21]:
train_trl(qlora=True, output_dir=os.path.join(OUTPUT_ROOT, "trl_qlora"), label="TRL + QLoRA")


=== TRL + QLoRA ===


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
401 Client Error. (Request ID: Root=1-69f1d213-130aaad92c8c5f343b8d713b;81cebe1b-fc96-487a-a05b-21ef5790d145)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

@torch.inference_mode()
def infer_trl(adapter_dir: str, qlora: bool, max_new_tokens: int = 256):
    """
    Tải model và adapter để thực hiện suy luận (Inference).
    """
    clear_gpu()

    # 1. Tải Tokenizer từ thư mục adapter (đã lưu ở bước train)
    tok = AutoTokenizer.from_pretrained(
        adapter_dir, 
        trust_remote_code=True, 
        token=HF_TOKEN
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # 2. Tải Base Model (Mô hình gốc)
    if qlora:
        # Nếu là QLoRA, phải tải base model ở dạng 4-bit đúng như lúc train
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=DTYPE,
            bnb_4bit_use_double_quant=True,
        )
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=True,
            token=HF_TOKEN,
        )
    else:
        # Load bình thường với kiểu dữ liệu DTYPE (float16 hoặc bfloat16)
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=DTYPE,
            device_map="auto",
            trust_remote_code=True,
            token=HF_TOKEN,
        )

    # 3. Kết hợp Base Model với Adapter (LoRA weights)
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()               # Chuyển sang chế độ dự đoán
    model.config.use_cache = True # Bật cache để tăng tốc độ sinh văn bản
    device = next(model.parameters()).device

    # 4. Chạy suy luận trên các mẫu INFER_SAMPLES
    outs = []
    for item in INFER_SAMPLES:
        msgs = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": item["question"]},
        ]
        
        # Áp dụng Chat Template và thêm token gợi ý cho Assistant trả lời
        prompt = tok.apply_chat_template(
            msgs, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        # Mã hóa văn bản và đưa vào GPU
        inp = tok(prompt, return_tensors="pt").to(device)
        
        # Sinh câu trả lời (Generation)
        out = model.generate(
            **inp,
            max_new_tokens=max_new_tokens,
            do_sample=False,       # Greedy decoding để kết quả ổn định
            pad_token_id=tok.eos_token_id,
        )
        
        # Giải mã và lọc lấy phần trả lời của Assistant
        decoded_text = tok.decode(out[0], skip_special_tokens=True)
        outs.append(extract_answer(decoded_text))

    # 5. Dọn dẹp bộ nhớ
    del model, base
    clear_gpu()
    
    return outs

In [ ]:
# --- Thực hiện suy luận (Inference) cho cả hai cấu hình ---

# 1. Thử nghiệm với mô hình LoRA (thông thường)
try:
    infer_results["TRL + LoRA"] = infer_trl(
        os.path.join(OUTPUT_ROOT, "trl_lora", "adapter"),
        qlora=False
    )
    print("[OK] TRL + LoRA")
except Exception as e:
    infer_results["TRL + LoRA"] = None
    print(f"[FAIL] TRL + LoRA\n{e}")

# 2. Thử nghiệm với mô hình QLoRA (4-bit)
try:
    infer_results["TRL + QLoRA"] = infer_trl(
        os.path.join(OUTPUT_ROOT, "trl_qlora", "adapter"),
        qlora=True
    )
    print("[OK] TRL + QLoRA")
except Exception as e:
    infer_results["TRL + QLoRA"] = None
    print(f"[FAIL] TRL + QLoRA\n{e}")